### Import packages and define paths

In [284]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
from gensim.models.word2vec import Word2Vec
from gensim.matutils import unitvec

In [285]:
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
EMBEDDINGS_DIR = PROJECT_ROOT / "embeddings"
DATA_DIR = PROJECT_ROOT / "data/dictionaries"
GENDERED_NAMES_DIR = PROJECT_ROOT / "data/gendered_names"
TARGET_WORDS_DIR = PROJECT_ROOT / "data/target_words"
OUTPUT_DIR = PROJECT_ROOT / "data/associational_gender_bias"

warmth_competence_lexicon_path = DATA_DIR / "Seed Dictionaries.csv"

lexicon = pd.read_csv(warmth_competence_lexicon_path)

In [286]:
warmth_competence_lexicon_path = DATA_DIR / "Seed Dictionaries.csv"

lexicon = pd.read_csv(warmth_competence_lexicon_path)

### Load embeddings

In [287]:
romance_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "romance.w2v"))
sci_fi_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "sci_fi.w2v"))
literary_fiction_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "literary_fiction.w2v"))

### Create average gender vectors

In [288]:
# Load previously extracted gendered names
with open(GENDERED_NAMES_DIR / "male_names_sci_fi.json") as f:
    male_names_sci_fi = json.load(f)

with open(GENDERED_NAMES_DIR / "female_names_sci_fi.json") as f:
    female_names_sci_fi = json.load(f)

with open(GENDERED_NAMES_DIR / "male_names_lit.json") as f:
    male_names_lit = json.load(f)

with open(GENDERED_NAMES_DIR / "female_names_lit.json") as f:
    female_names_lit = json.load(f)

with open(GENDERED_NAMES_DIR / "male_names_romance.json") as f:
    male_names_romance = json.load(f)
    
with open(GENDERED_NAMES_DIR / "female_names_romance.json") as f:
    female_names_romance = json.load(f)

In [289]:
female_list = ['she', 'daughter', 'hers', 'her', 'mother', 'woman', 'girl', 'herself', 'female', 'sister', 'daughters', 'mothers', 'women', 'girls', 'females', 'sisters', 'aunt', 'aunts', 'niece', 'nieces']
male_list = ['he', 'son', 'his', 'him', 'father', 'man', 'boy', 'himself', 'male', 'brother', 'sons', 'fathers', 'men', 'boys', 'males', 'brothers', 'uncle', 'uncles', 'nephew', 'nephews']

In [290]:
female_list_sci_fi = female_list + female_names_sci_fi
male_list_sci_fi = male_list + male_names_sci_fi

female_list_lit = female_list + female_names_lit
male_list_lit = male_list + male_names_lit

female_list_romance = female_list + female_names_romance 
male_list_romance = male_list + male_names_romance

In [291]:
# Function to compute the average vector
def compute_average_vector(model, word_list):
    vectors = [model.wv[word] for word in word_list if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        raise ValueError(f"None of the words in {word_list} exist in the model.")

In [292]:
def cosine_sim(a, b):
    return np.dot(unitvec(a), unitvec(b))

In [293]:
def update_gender_vectors(model, male_words, female_words, genre_name, save_dir):
    # Save original vectors
    original_he = model.wv["he"].copy()
    original_she = model.wv["she"].copy()

    # Compute new vectors
    he_vector = compute_average_vector(model, male_words)
    she_vector = compute_average_vector(model, female_words)

    # Safe update
    model.wv["he"] = he_vector
    model.wv["she"] = she_vector

    # Compare with original
    he_sim = cosine_sim(original_he, model.wv["he"])
    she_sim = cosine_sim(original_she, model.wv["she"])
    print(f"{genre_name} — he similarity with original: {he_sim:.6f}")
    print(f"{genre_name} — she similarity with original: {she_sim:.6f}")

    # Save the updated model to a new file
    save_path = save_dir / f"{genre_name}_with_gender_vectors.w2v"
    model.save(str(save_path))

    return model

In [294]:
# With names
update_gender_vectors(romance_embeddings, male_list_romance, female_list_romance, "romance", EMBEDDINGS_DIR)
update_gender_vectors(literary_fiction_embeddings, male_list_lit, female_list_lit, "literary_fiction", EMBEDDINGS_DIR)
update_gender_vectors(sci_fi_embeddings, male_list_sci_fi, female_list_sci_fi, "sci_fi", EMBEDDINGS_DIR)

updated_romance_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "romance_with_gender_vectors.w2v"))
updated_sci_fi_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "sci_fi_with_gender_vectors.w2v"))
updated_literary_fiction_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "literary_fiction_with_gender_vectors.w2v"))


romance — he similarity with original: 0.668137
romance — she similarity with original: 0.657199
literary_fiction — he similarity with original: 0.675241
literary_fiction — she similarity with original: 0.655275
sci_fi — he similarity with original: 0.654382
sci_fi — she similarity with original: 0.623882


In [295]:
# Without names

# Reload original embeddings so we start from a clean baseline
romance_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "romance.w2v"))
literary_fiction_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "literary_fiction.w2v"))
sci_fi_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "sci_fi.w2v"))

update_gender_vectors(romance_embeddings, male_list, female_list, "romance_no_names", EMBEDDINGS_DIR)
update_gender_vectors(literary_fiction_embeddings, male_list, female_list, "literary_fiction_no_names", EMBEDDINGS_DIR)
update_gender_vectors(sci_fi_embeddings, male_list, female_list, "sci_fi_no_names", EMBEDDINGS_DIR)

updated_romance_no_names_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "romance_no_names_with_gender_vectors.w2v"))
updated_sci_fi_no_names_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "sci_fi_no_names_with_gender_vectors.w2v"))
updated_literary_fiction_no_names_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "literary_fiction_no_names_with_gender_vectors.w2v"))

romance_no_names — he similarity with original: 0.604026
romance_no_names — she similarity with original: 0.545454
literary_fiction_no_names — he similarity with original: 0.633374
literary_fiction_no_names — she similarity with original: 0.584341
sci_fi_no_names — he similarity with original: 0.676018
sci_fi_no_names — she similarity with original: 0.604890


### Define target dimensions/word categories

In [296]:
high_sociability_words = lexicon[(lexicon["Dictionary"] == "Sociability") & (lexicon["Dir"] == "high")]["term"].tolist()
low_sociability_words = lexicon[(lexicon["Dictionary"] == "Sociability") & (lexicon["Dir"] == "low")]["term"].tolist()
high_morality_words = lexicon[(lexicon["Dictionary"] == "Morality") & (lexicon["Dir"] == "high")]["term"].tolist()
low_morality_words = lexicon[(lexicon["Dictionary"] == "Morality") & (lexicon["Dir"] == "low")]["term"].tolist()

high_agency_words = lexicon[(lexicon["Dictionary"] == "Agency") & (lexicon["Dir"] == "high")]["term"].tolist()
low_agency_words = lexicon[(lexicon["Dictionary"] == "Agency") & (lexicon["Dir"] == "low")]["term"].tolist()
high_ability_words = lexicon[(lexicon["Dictionary"] == "Ability") & (lexicon["Dir"] == "high")]["term"].tolist()
low_ability_words = lexicon[(lexicon["Dictionary"] == "Ability") & (lexicon["Dir"] == "low")]["term"].tolist()

In [297]:
# Remove duplicates within each combined list, preserving order
high_warmth     = list(dict.fromkeys(high_sociability_words + high_morality_words))
low_warmth      = list(dict.fromkeys(low_sociability_words + low_morality_words))
high_competence = list(dict.fromkeys(high_agency_words + high_ability_words))
low_competence  = list(dict.fromkeys(low_agency_words + low_ability_words))

In [298]:
len(high_competence), len(low_competence), len(high_warmth), len(low_warmth)

(68, 60, 74, 81)

### Check missing words and make specific category list for each genre depending on missing words

In [299]:
genre_embeddings = {
    "romance": romance_embeddings,
    "sci_fi": sci_fi_embeddings,
    "literary_fiction": literary_fiction_embeddings
}

word_lists = {
    "high_competence": high_competence,
    "low_competence": low_competence,
    "high_warmth": high_warmth,
    "low_warmth": low_warmth
}
# Function to create genre-specific filtered lists
def create_genre_specific_lists(genre_embeddings, word_lists):
    filtered_lists = {}
    missing_summary = {}
    
    for genre_name, emb in genre_embeddings.items():
        for category_name, words in word_lists.items():
            # Filter words present in this genre's embeddings
            filtered = [w for w in words if w in emb.wv]
            missing = [w for w in words if w not in emb.wv]
            
            # Key name format: category_genre
            key = f"{category_name}_{genre_name}"
            filtered_lists[key] = filtered
            missing_summary[key] = missing
            
    return filtered_lists, missing_summary


In [300]:
# Apply the function
filtered_genre_lists, missing_genre_words = create_genre_specific_lists(genre_embeddings, word_lists)

# access the filtered lists
high_competence_romance = filtered_genre_lists["high_competence_romance"]
high_competence_sci_fi = filtered_genre_lists["high_competence_sci_fi"]
high_competence_literary_fiction = filtered_genre_lists["high_competence_literary_fiction"]

low_competence_romance = filtered_genre_lists["low_competence_romance"]
low_competence_sci_fi = filtered_genre_lists["low_competence_sci_fi"]
low_competence_literary_fiction = filtered_genre_lists["low_competence_literary_fiction"]

high_warmth_romance = filtered_genre_lists["high_warmth_romance"]
high_warmth_sci_fi = filtered_genre_lists["high_warmth_sci_fi"]
high_warmth_literary_fiction = filtered_genre_lists["high_warmth_literary_fiction"]

low_warmth_romance = filtered_genre_lists["low_warmth_romance"]
low_warmth_sci_fi = filtered_genre_lists["low_warmth_sci_fi"]
low_warmth_literary_fiction = filtered_genre_lists["low_warmth_literary_fiction"]

# inspect missing words
for key, missing in missing_genre_words.items():
    if missing:
        print(f"{key}: {len(missing)} missing words -> {missing}")

high_competence_romance: 28 missing words -> ['fearlessness', 'assertiveness', 'assertive', 'unassertive', 'striver', 'striving', 'industrious', 'energetic', 'self-confident', 'self-reliant', 'conscientious', 'motivated', 'unwavering', 'autonomous', 'dominating', 'dominant', 'dominance', 'competitive', 'smart', 'intelligent', 'skillful', 'educated', 'felicitous', 'imaginative', 'shrewd', 'discriminating', 'inventive', 'insightful']
low_competence_romance: 41 missing words -> ['diffident', 'unassertiveness', 'insecure', 'inactive', 'doubtful', 'dependent', 'apathy', 'unenterprising', 'negligent', 'lethargic', 'unambitious', 'undedicated', 'unadventurous', 'unmotivated', 'nonresilient', 'spiritless', 'dominated', 'submissive', 'meek', 'docile', 'incompetent', 'uncompetitive', 'unintelligent', 'stupidity', 'ignorant', 'ignorance', 'dumb', 'dumbness', 'uneducated', 'uncreative', 'incapable', 'unimaginative', 'undiscriminating', 'maladroit', 'folly', 'unwise', 'inefficient', 'ineffective', 

In [301]:
# Combine categories
all_categories_romance = {
    "high_competence": high_competence_romance,
    "low_competence": low_competence_romance,
    "high_warmth": high_warmth_romance,
    "low_warmth": low_warmth_romance
}

# save as json
with open(TARGET_WORDS_DIR / "target_words_romance.json", "w") as f:
    json.dump(all_categories_romance, f)


all_categories_sci_fi = {
    "high_competence": high_competence_sci_fi,
    "low_competence": low_competence_sci_fi,
    "high_warmth": high_warmth_sci_fi,
    "low_warmth": low_warmth_sci_fi
}

# save as json
with open(TARGET_WORDS_DIR / "target_words_sci_fi.json", "w") as f:
    json.dump(all_categories_sci_fi, f)

all_categories_literary_fiction = {
    "high_competence": high_competence_literary_fiction,
    "low_competence": low_competence_literary_fiction,
    "high_warmth": high_warmth_literary_fiction,
    "low_warmth": low_warmth_literary_fiction
}

# save as json
with open(TARGET_WORDS_DIR / "target_words_literary_fiction.json", "w") as f:
    json.dump(all_categories_literary_fiction, f)

### Calculate similarity

In [302]:
# Function to calculate the cosine similarity using gensim's similarity method
def calculate_similarity(embeddings, word1, word2):
    if word1 in embeddings.wv and word2 in embeddings.wv:
        return embeddings.wv.similarity(word1, word2)
    else:
        return None  # if any word is missing

In [303]:
data_romance = []
data_romance_no_names = []
data_sci_fi = []
data_sci_fi_no_names = []
data_literary_fiction = []
data_literary_fiction_no_names = []

In [304]:
for dimensions, target_words in all_categories_romance.items():
        for target_word in target_words:
            # Calculate similarity with "he" and "she"
            similarity_he = calculate_similarity(updated_romance_embeddings, target_word, "he")
            similarity_she = calculate_similarity(updated_romance_embeddings, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she  # Gender bias as the difference
                
                data_romance.append({
                    "dimension": dimensions,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

In [305]:
for dimensions, target_words in all_categories_romance.items():
        for target_word in target_words:
            # Calculate similarity with "he" and "she"
            similarity_he = calculate_similarity(updated_romance_no_names_embeddings, target_word, "he")
            similarity_she = calculate_similarity(updated_romance_no_names_embeddings, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she  # Gender bias as the difference
                
                data_romance_no_names.append({
                    "dimension": dimensions,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

In [306]:
for dimensions, target_words in all_categories_sci_fi.items():
        for target_word in target_words:
            # Calculate similarity with "he" and "she"
            similarity_he = calculate_similarity(updated_sci_fi_embeddings, target_word, "he")
            similarity_she = calculate_similarity(updated_sci_fi_embeddings, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she  # Gender bias as the difference
                
                data_sci_fi.append({
                    "dimension": dimensions,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

In [307]:
for dimensions, target_words in all_categories_sci_fi.items():
        for target_word in target_words:
            # Calculate similarity with "he" and "she"
            similarity_he = calculate_similarity(updated_sci_fi_no_names_embeddings, target_word, "he")
            similarity_she = calculate_similarity(updated_sci_fi_no_names_embeddings, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she  # Gender bias as the difference
                
                data_sci_fi_no_names.append({
                    "dimension": dimensions,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

In [308]:
for dimensions, target_words in all_categories_literary_fiction.items():
        for target_word in target_words:
            # Calculate similarity with "he" and "she"
            similarity_he = calculate_similarity(updated_literary_fiction_embeddings, target_word, "he")
            similarity_she = calculate_similarity(updated_literary_fiction_embeddings, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she  # Gender bias as the difference
                
                data_literary_fiction.append({
                    "dimension": dimensions,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

In [309]:
for dimensions, target_words in all_categories_literary_fiction.items():
        for target_word in target_words:
            # Calculate similarity with "he" and "she"
            similarity_he = calculate_similarity(updated_literary_fiction_no_names_embeddings, target_word, "he")
            similarity_she = calculate_similarity(updated_literary_fiction_no_names_embeddings, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she  # Gender bias as the difference
                
                data_literary_fiction_no_names.append({
                    "dimension": dimensions,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

In [310]:
# Convert results into a pandas DataFrame
df_romance = pd.DataFrame(data_romance)
df_romance_no_names = pd.DataFrame(data_romance_no_names)
df_sci_fi = pd.DataFrame(data_sci_fi)
df_sci_fi_no_names = pd.DataFrame(data_sci_fi_no_names)
df_literary_fiction = pd.DataFrame(data_literary_fiction)
df_literary_fiction_no_names = pd.DataFrame(data_literary_fiction_no_names)

df_romance.to_csv( OUTPUT_DIR / "romance_associational_gender_bias.csv", index=False)
df_romance_no_names.to_csv( OUTPUT_DIR / "romance_no_names_associational_gender_bias.csv", index=False)
df_sci_fi.to_csv( OUTPUT_DIR / "sci_fi_associational_gender_bias.csv", index=False)
df_sci_fi_no_names.to_csv( OUTPUT_DIR / "sci_fi_no_names_associational_gender_bias.csv", index=False)
df_literary_fiction.to_csv( OUTPUT_DIR / "literary_fiction_associational_gender_bias.csv", index=False)
df_literary_fiction_no_names.to_csv( OUTPUT_DIR / "literary_fiction_no_names_associational_gender_bias.csv", index=False)

In [311]:
df_romance_no_names[["word", "gender_bias"]].sort_values(by="gender_bias", ascending=False).head(10)

,word,gender_bias
32,clever,0.090817
30,graceful,0.082978
23,skill,0.078597
21,intelligence,0.069123
119,bad,0.066756
113,rough,0.065266
132,unreliable,0.059572
69,sympathy,0.055026
89,good,0.054827
108,disliked,0.052952


In [312]:
df_romance[["word", "gender_bias"]].sort_values(by="gender_bias", ascending=False).head(10)

,word,gender_bias
74,civil,0.052638
36,effective,0.047897
113,rough,0.046761
77,nice,0.044506
68,sympathetic,0.039733
104,reliable,0.038760
119,bad,0.037887
32,clever,0.036433
17,aggressive,0.032981
69,sympathy,0.031817


In [313]:
df_sci_fi_no_names[["word", "gender_bias"]].sort_values(by="gender_bias", ascending=False).head(10)

,word,gender_bias
2,confidence,0.094865
146,unreliable,0.073985
1,confident,0.072825
50,careless,0.071905
130,threat,0.066258
122,disliked,0.064952
52,helpless,0.058598
131,bad,0.055920
73,liked,0.055292
138,liar,0.052357


In [314]:
df_sci_fi[["word", "gender_bias"]].sort_values(by="gender_bias", ascending=False).head(10)

,word,gender_bias
86,funny,0.033591
138,liar,0.026795
131,bad,0.025578
56,stupid,0.022879
11,impulsive,0.022602
50,careless,0.020739
35,clever,0.020059
76,affectionate,0.018997
1,confident,0.017904
2,confidence,0.016463


In [315]:
df_literary_fiction_no_names[["word", "gender_bias"]].sort_values(by="gender_bias", ascending=False).head(10)

,word,gender_bias
48,helpless,0.101518
131,cunning,0.084297
31,clever,0.083392
117,rough,0.080041
107,reliable,0.071891
97,sincerity,0.071887
24,skilled,0.070641
52,ignorance,0.065728
65,liked,0.063381
30,graceful,0.062832


In [316]:
df_literary_fiction[["word", "gender_bias"]].sort_values(by="gender_bias", ascending=False).head(10)

,word,gender_bias
97,sincerity,0.076734
31,clever,0.065840
117,rough,0.065260
76,funny,0.063755
19,smart,0.062652
107,reliable,0.061416
48,helpless,0.059060
122,bad,0.057624
24,skilled,0.053655
52,ignorance,0.050634
